In [2]:
# ==============================
# Step 1: Create Salary Category Target
# ==============================

import pandas as pd
import numpy as np

# Fresh load (new notebook, independent kernel)
df = pd.read_csv("../data/raw/salary.csv")

# Create 3 balanced categories using quantile-based binning
df["salary_category"] = pd.qcut(
    df["salary"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Verify class balance
print(df["salary_category"].value_counts())
print("\nPercentage distribution:")
print(df["salary_category"].value_counts(normalize=True) * 100)

# Check the actual salary boundaries used
print("\nBin edges:")
print(pd.qcut(df["salary"], q=3).cat.categories)

salary_category
Low       83334
Medium    83333
High      83333
Name: count, dtype: int64

Percentage distribution:
salary_category
Low       33.3336
Medium    33.3332
High      33.3332
Name: proportion, dtype: float64

Bin edges:
IntervalIndex([(31866.999, 127849.0], (127849.0, 159802.0],
               (159802.0, 333046.0]],
              dtype='interval[float64, right]')


In [3]:
# ==============================
# Step 2: Separate Features (X) and Target (y)
# ==============================

# Target: the new category column
y = df["salary_category"]

# Features: drop BOTH salary and salary_category
X = df.drop(columns=["salary", "salary_category"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:", X.columns.tolist())

X shape: (250000, 9)
y shape: (250000,)

Feature columns: ['job_title', 'experience_years', 'education_level', 'skills_count', 'industry', 'company_size', 'location', 'remote_work', 'certifications']


In [4]:
# ==============================
# Step 3: Encode Target Labels
# ==============================

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes found:", label_encoder.classes_)
print("Sample original y:", y[:5].tolist())
print("Sample encoded y:", y_encoded[:5])

Classes found: ['High' 'Low' 'Medium']
Sample original y: ['Low', 'Low', 'Medium', 'High', 'High']
Sample encoded y: [1 1 2 0 0]


In [5]:
# ==============================
# Step 4: Train-Test Split
# ==============================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Confirm class balance is preserved in both sets
import numpy as np
print("\nTrain class distribution:", np.bincount(y_train))
print("Test class distribution:", np.bincount(y_test))

X_train shape: (200000, 9)
X_test shape: (50000, 9)
y_train shape: (200000,)
y_test shape: (50000,)

Train class distribution: [66666 66667 66667]
Test class distribution: [16667 16667 16666]


In [6]:
# ==============================
# Step 5: Preprocess Features
# ==============================

import sys
sys.path.append("..")

from src.preprocess import build_preprocessor

preprocessor = build_preprocessor()

# Fit ONLY on training data
preprocessor.fit(X_train)

# Transform both sets
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

# Save this classification-specific preprocessor separately
import joblib
import os
os.makedirs("../models", exist_ok=True)
joblib.dump(preprocessor, "../models/preprocessor_classifier.pkl")

print("Classification preprocessor saved successfully.")

X_train_processed shape: (200000, 38)
X_test_processed shape: (50000, 38)
Classification preprocessor saved successfully.


In [7]:
# ==============================
# Step 6: Train the Logistic Regression Model
# ==============================

from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train_processed, y_train)

print("Classifier trained successfully.")
print("Classes:", classifier.classes_)
print("Number of coefficient sets:", classifier.coef_.shape)

Classifier trained successfully.
Classes: [0 1 2]
Number of coefficient sets: (3, 38)


In [8]:
# ==============================
# Step 7: Evaluate the Classifier
# ==============================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Predict on test set
y_pred = classifier.predict(X_test_processed)

# Overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy, 4))

# Precision, Recall, F1 (macro average = treats all classes equally)
precision = precision_score(y_test, y_pred, average="macro")
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

print("Precision (macro):", round(precision, 4))
print("Recall (macro):", round(recall, 4))
print("F1-score (macro):", round(f1, 4))

# Detailed per-class report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9061
Precision (macro): 0.9065
Recall (macro): 0.9061
F1-score (macro): 0.9063

Classification Report:
              precision    recall  f1-score   support

        High       0.94      0.93      0.93     16667
         Low       0.93      0.92      0.93     16667
      Medium       0.85      0.87      0.86     16666

    accuracy                           0.91     50000
   macro avg       0.91      0.91      0.91     50000
weighted avg       0.91      0.91      0.91     50000

Confusion Matrix:
[[15477     0  1190]
 [    0 15384  1283]
 [ 1058  1163 14445]]


In [9]:
# ==============================
# Step 8: Save the Classifier
# ==============================

import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(classifier, "../models/logistic_regression_model.pkl")
joblib.dump(label_encoder, "../models/label_encoder.pkl")

print("Classifier saved to ../models/logistic_regression_model.pkl")
print("Label encoder saved to ../models/label_encoder.pkl")

Classifier saved to ../models/logistic_regression_model.pkl
Label encoder saved to ../models/label_encoder.pkl
